## Exercise 3

In [17]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, date_format, hour, round, to_date, when
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

spark = (
    SparkSession.builder
    .appName("Session09Part03")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

schema = StructType([
    StructField("event_id", IntegerType(), True),
    StructField("service", StringType(), True),
    StructField("region", StringType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("request_count", IntegerType(), True),
    StructField("error_count", IntegerType(), True),
    StructField("latency_ms", DoubleType(), True),
    StructField("bytes_in", DoubleType(), True),
    StructField("bytes_out", DoubleType(), True),
])

events_df = spark.read.option("header", True).schema(schema).csv("../datasets/service_events.csv")

events_enriched_df = (
    events_df
    .withColumn("error_rate", when(col("request_count") > 0, col("error_count") / col("request_count")).otherwise(0))
    .withColumn("total_bytes", col("bytes_in") + col("bytes_out"))
    .withColumn("traffic_mb", col("total_bytes") / 1048576)
    .withColumn(
        "latency_band",
        when(col("latency_ms") < 100, "fast")
        .when(col("latency_ms") < 160, "normal")
        .otherwise("slow"),
    )
    .withColumn("event_date", to_date(col("event_time")))
    .withColumn("event_hour", hour(col("event_time")))
    .withColumn("day_of_week", date_format(col("event_time"), "E"))
)

In [18]:
from pyspark.sql.functions import avg, count, stddev, sum

service_summary_df = events_enriched_df.groupBy("service").agg(
    count("*").alias("total_records"),
    sum("request_count").alias("total_requests"),
    sum("error_count").alias("total_errors"),
    avg("error_rate").alias("average_error_rate"),
    avg("latency_ms").alias("average_latency_ms"),
    stddev("latency_ms").alias("latency_stddev"),
    sum("traffic_mb").alias("total_traffic_mb"),
)

service_summary_df.show()

+---------------+-------------+--------------+------------+--------------------+------------------+------------------+-----------------+
|        service|total_records|total_requests|total_errors|  average_error_rate|average_latency_ms|    latency_stddev| total_traffic_mb|
+---------------+-------------+--------------+------------+--------------------+------------------+------------------+-----------------+
|           auth|            6|          7580|          39|0.005108422813243081| 79.63333333333334| 4.147609753420242|305.5572509765625|
|recommendations|            6|          9880|         119|0.011924216567787992|140.75000000000003| 16.61791202287459|723.6480712890625|
|       payments|            6|          5590|         136| 0.02432688314389142|174.63333333333333|22.910841683942277|407.8865051269531|
|         search|            6|         13080|          75|0.005716708875830474| 94.01666666666667| 5.659475829674217|614.2616271972656|
+---------------+-------------+----------

In [19]:
from pyspark.sql.functions import dense_rank
from pyspark.sql.window import Window

traffic_window = Window.orderBy(col("total_traffic_mb").desc())
variability_window = Window.orderBy(col("latency_stddev").desc())
reliability_window = Window.orderBy(col("average_error_rate").asc())

ranked_summary_df = (
    service_summary_df
    .withColumn("traffic_rank", dense_rank().over(traffic_window))
    .withColumn("latency_variability_rank", dense_rank().over(variability_window))
    .withColumn("reliability_rank", dense_rank().over(reliability_window))
)

ranked_summary_df.orderBy("traffic_rank").show()

+---------------+-------------+--------------+------------+--------------------+------------------+------------------+-----------------+------------+------------------------+----------------+
|        service|total_records|total_requests|total_errors|  average_error_rate|average_latency_ms|    latency_stddev| total_traffic_mb|traffic_rank|latency_variability_rank|reliability_rank|
+---------------+-------------+--------------+------------+--------------------+------------------+------------------+-----------------+------------+------------------------+----------------+
|recommendations|            6|          9880|         119|0.011924216567787992|140.75000000000003| 16.61791202287459|723.6480712890625|           1|                       2|               3|
|         search|            6|         13080|          75|0.005716708875830474| 94.01666666666667| 5.659475829674217|614.2616271972656|           2|                       3|               2|
|       payments|            6|         

In [20]:
hourly_requests_df = events_enriched_df.groupBy("service", "event_hour").agg(
    sum("request_count").alias("hourly_requests")
)

hour_window = Window.partitionBy("service").orderBy(col("hourly_requests").desc())

busiest_hour_df = (
    hourly_requests_df
    .withColumn("hour_rank", dense_rank().over(hour_window))
    .filter(col("hour_rank") == 1)
    .select("service", col("event_hour").alias("busiest_hour"))
)

busiest_hour_df.show()

+---------------+------------+
|        service|busiest_hour|
+---------------+------------+
|           auth|           9|
|       payments|           9|
|recommendations|           3|
|         search|           3|
+---------------+------------+



In [21]:
final_summary_df = (
    ranked_summary_df
    .join(busiest_hour_df, on="service", how="left")
    .select(
        "service",
        "total_records",
        "total_requests",
        "total_errors",
        round("average_error_rate", 4).alias("average_error_rate"),
        round("average_latency_ms", 2).alias("average_latency_ms"),
        round("latency_stddev", 2).alias("latency_stddev"),
        round("total_traffic_mb", 2).alias("total_traffic_mb"),
        "busiest_hour",
        "traffic_rank",
        "latency_variability_rank",
        "reliability_rank",
    )
    .orderBy("traffic_rank")
)

final_summary_df.show(truncate=False)

+---------------+-------------+--------------+------------+------------------+------------------+--------------+----------------+------------+------------+------------------------+----------------+
|service        |total_records|total_requests|total_errors|average_error_rate|average_latency_ms|latency_stddev|total_traffic_mb|busiest_hour|traffic_rank|latency_variability_rank|reliability_rank|
+---------------+-------------+--------------+------------+------------------+------------------+--------------+----------------+------------+------------+------------------------+----------------+
|recommendations|6            |9880          |119         |0.0119            |140.75            |16.62         |723.65          |3           |1           |2                       |3               |
|search         |6            |13080         |75          |0.0057            |94.02             |5.66          |614.26          |3           |2           |3                       |2               |
|payments 

In [22]:
final_summary_df.coalesce(1).write.mode("overwrite").option("header", True).csv("results/service_summary_spark")

In [23]:
from pathlib import Path
import shutil

output_folder = Path("results/service_summary_spark")
single_csv = Path("results/service_summary.csv")

part_file = next(output_folder.glob("part-*.csv"))
shutil.copy(part_file, single_csv)

print(f"Saved {single_csv}")

Saved results/service_summary.csv


### Exercises:

In [24]:
final_summary_df.show()

+---------------+-------------+--------------+------------+------------------+------------------+--------------+----------------+------------+------------+------------------------+----------------+
|        service|total_records|total_requests|total_errors|average_error_rate|average_latency_ms|latency_stddev|total_traffic_mb|busiest_hour|traffic_rank|latency_variability_rank|reliability_rank|
+---------------+-------------+--------------+------------+------------------+------------------+--------------+----------------+------------+------------+------------------------+----------------+
|recommendations|            6|          9880|         119|            0.0119|            140.75|         16.62|          723.65|           3|           1|                       2|               3|
|         search|            6|         13080|          75|            0.0057|             94.02|          5.66|          614.26|           3|           2|                       3|               2|
|       pa

In [25]:
final_summary_df.filter(final_summary_df["traffic_rank"] == 1).show()

+---------------+-------------+--------------+------------+------------------+------------------+--------------+----------------+------------+------------+------------------------+----------------+
|        service|total_records|total_requests|total_errors|average_error_rate|average_latency_ms|latency_stddev|total_traffic_mb|busiest_hour|traffic_rank|latency_variability_rank|reliability_rank|
+---------------+-------------+--------------+------------+------------------+------------------+--------------+----------------+------------+------------+------------------------+----------------+
|recommendations|            6|          9880|         119|            0.0119|            140.75|         16.62|          723.65|           3|           1|                       2|               3|
+---------------+-------------+--------------+------------+------------------+------------------+--------------+----------------+------------+------------+------------------------+----------------+



In [26]:
final_summary_df.filter(final_summary_df["latency_variability_rank"] == 1).show()

+--------+-------------+--------------+------------+------------------+------------------+--------------+----------------+------------+------------+------------------------+----------------+
| service|total_records|total_requests|total_errors|average_error_rate|average_latency_ms|latency_stddev|total_traffic_mb|busiest_hour|traffic_rank|latency_variability_rank|reliability_rank|
+--------+-------------+--------------+------------+------------------+------------------+--------------+----------------+------------+------------+------------------------+----------------+
|payments|            6|          5590|         136|            0.0243|            174.63|         22.91|          407.89|           9|           3|                       1|               4|
+--------+-------------+--------------+------------+------------------+------------------+--------------+----------------+------------+------------+------------------------+----------------+



In [28]:
final_summary_df.filter(final_summary_df["reliability_rank"] == 1).show()

+-------+-------------+--------------+------------+------------------+------------------+--------------+----------------+------------+------------+------------------------+----------------+
|service|total_records|total_requests|total_errors|average_error_rate|average_latency_ms|latency_stddev|total_traffic_mb|busiest_hour|traffic_rank|latency_variability_rank|reliability_rank|
+-------+-------------+--------------+------------+------------------+------------------+--------------+----------------+------------+------------+------------------------+----------------+
|   auth|            6|          7580|          39|            0.0051|             79.63|          4.15|          305.56|           9|           4|                       4|               1|
+-------+-------------+--------------+------------+------------------+------------------+--------------+----------------+------------+------------+------------------------+----------------+



In [29]:
spark.stop()